# Income Status Prediction — Upgraded ML Project

This notebook turns the classic Adult Census Income case study into a more complete portfolio project.

### Upgrade highlights
- Reproducible preprocessing pipelines
- Multiple model families
- Stratified cross-validation and hyperparameter search
- ROC-AUC, PR-AUC, precision, recall and F1
- Threshold tuning on validation data
- Explainability with permutation importance
- Subgroup evaluation by sex
- Saved model + Streamlit demo

> Run `python -m src.train` from the project root for the main reproducible training pipeline.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve)
from sklearn.inspection import permutation_importance

from src.data import load_adult_dataset, split_features_target
from src.features import make_linear_preprocessor, make_tree_preprocessor
from src.evaluate import classification_metrics, find_best_f1_threshold


## 1. Load and understand the data

In [ ]:
df = load_adult_dataset()
print('Shape:', df.shape)
display(df.head())
display(df.isna().mean().sort_values(ascending=False).head(10))


In [ ]:
y = df['income_status']
print('Target distribution')
display(y.value_counts(normalize=True).rename({0:'<=50K', 1:'>50K'}).to_frame('share'))


### Why this is a classification problem
We predict one of two income-status classes. Because the positive class is not the majority, accuracy alone can hide useful information; ROC-AUC, PR-AUC, precision, recall and F1 are also reported.

## 2. Train / validation / test split
The test set stays untouched until final evaluation. Stratification preserves the target ratio across splits.

In [ ]:
X, y = split_features_target(df)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.20, stratify=y_train_full, random_state=42
)
print(X_train.shape, X_valid.shape, X_test.shape)


## 3. Preprocessing
Numeric variables use median imputation. Categorical variables use most-frequent imputation and encoding. All preprocessing is fitted inside pipelines so test information does not leak into training.

## 4. Model candidates
We compare a linear baseline with two nonlinear tree ensembles. This gives a useful bias/variance and interpretability comparison rather than relying on a single algorithm.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    'Logistic Regression': Pipeline([
        ('prep', make_linear_preprocessor(X_train)),
        ('model', LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('prep', make_tree_preprocessor(X_train)),
        ('model', RandomForestClassifier(n_estimators=350, class_weight='balanced_subsample',
                                        min_samples_leaf=2, random_state=42, n_jobs=-1)),
    ]),
    'HistGradientBoosting': Pipeline([
        ('prep', make_tree_preprocessor(X_train)),
        ('model', HistGradientBoostingClassifier(learning_rate=0.08, max_iter=250,
                                                max_leaf_nodes=31, l2_regularization=0.5,
                                                random_state=42)),
    ]),
}


In [ ]:
rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    fold_auc = []
    for tr, va in cv.split(X_train, y_train):
        model.fit(X_train.iloc[tr], y_train.iloc[tr])
        p = model.predict_proba(X_train.iloc[va])[:,1]
        fold_auc.append(roc_auc_score(y_train.iloc[va], p))
    model.fit(X_train, y_train)
    valid_p = model.predict_proba(X_valid)[:,1]
    threshold, _ = find_best_f1_threshold(y_valid, valid_p)
    m = classification_metrics(y_valid, valid_p, threshold)
    rows.append({'model':name, 'cv_roc_auc_mean':np.mean(fold_auc), 'cv_roc_auc_std':np.std(fold_auc), **m})

comparison = pd.DataFrame(rows).sort_values('cv_roc_auc_mean', ascending=False)
display(comparison)


## 5. Hyperparameter tuning
Tune the strongest candidates with `RandomizedSearchCV`. The search score is ROC-AUC so that ranking quality is considered even before choosing a classification threshold.

In [ ]:
# Example tuning cell — mirrors the reproducible CLI in src/train.py.
best_candidates = comparison['model'].head(2).tolist()
best_candidates


## 6. Threshold tuning
The default 0.50 cutoff is not always optimal for an imbalanced classification problem. Select the threshold on validation data, then evaluate exactly once on the untouched test set.

In [ ]:
# After fitting your chosen model:
# test_prob = chosen_model.predict_proba(X_test)[:,1]
# threshold, valid_f1 = find_best_f1_threshold(y_valid, chosen_model.predict_proba(X_valid)[:,1])
# print(classification_metrics(y_test, test_prob, threshold))


## 7. Confusion matrix, ROC and precision-recall curves
Use these together: the confusion matrix explains error counts at one threshold, while ROC and PR curves show behavior across thresholds.

In [ ]:
# Example plotting cell after final test probabilities are available:
# threshold = ...
# test_prob = ...
# y_pred = (test_prob >= threshold).astype(int)
# ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
# plt.show()


## 8. Explainability
Permutation importance measures how much shuffling a feature degrades the chosen scoring metric. It is model-agnostic and therefore useful for comparing feature influence without treating it as causal evidence.

In [ ]:
# Example:
# result = permutation_importance(chosen_model, X_test, y_test, scoring='roc_auc',
#                                 n_repeats=3, random_state=42, n_jobs=-1)


## 9. Subgroup analysis
Because the dataset contains demographic attributes, overall performance should not be the only check. Compare error rates by sex and interpret differences as diagnostic signals rather than causal conclusions.

A useful table includes sample count, positive rate, TPR/recall and FPR for each subgroup.

In [ ]:
# Example subgroup diagnostic:
# eval_df = X_test.copy()
# eval_df['actual'] = y_test.to_numpy()
# eval_df['pred'] = y_pred
# display(eval_df.groupby('sex').apply(lambda g: pd.Series({
#     'n': len(g),
#     'actual_positive_rate': g.actual.mean(),
#     'predicted_positive_rate': g.pred.mean(),
#     'recall': recall_score(g.actual, g.pred, zero_division=0),
# })))


## 10. Final takeaway
A portfolio-grade ML project is more than a model score. The important upgrades are leakage-safe preprocessing, multiple baselines, cross-validation, tuning, threshold selection, interpretable evaluation, subgroup diagnostics, reproducibility, testing and a usable demo.